# GRACE: Gated Rationale-Aware Cross-modal Encoder
## Hateful Meme Detection — B.Tech Major Project Notebook

End-to-end implementation of the GRACE methodology on three benchmarks:

| Dataset | Size | Task | Source |
|---|---|---|---|
| Facebook Hateful Memes Challenge (HMC) | ~10,000 | Binary hate | Meta / DrivenData |
| HarMeme | 3,544 | Harmful (COVID-19) | LCS2-IIITD |
| PrideMM | 5,063 | Hate (LGBTQ+) | Shah et al. 2024 |

**Target hardware:** NVIDIA T4 (Kaggle free tier, 16 GB VRAM).

**Architecture summary**
1. **Stage 1 (offline)** — extract CLIP ViT-B/32 features; LLaVA-1.5-7B (4-bit) generates 10 visual keywords + 1 rationale per meme; both are CLIP-text-encoded and cached.
2. **Stage 2** — ATIN (MHSA with learnable query) is trained independently on the keyword task, then frozen to produce `z_tag`.
3. **Stage 3** — RDM (2-layer MLP) is trained with NT-Xent contrastive loss against `z_tag`, then frozen to produce `d`.
4. **Stage 4** — Lightweight GRACE model (modality adapters + DPCAF + ArcFace head + dual DSR reconstruction heads) is trained on precomputed embeddings. Only Stage 4 runs on every training pass; Stages 1–3 are one-off.

**Total loss:**
$$\mathcal{L} = (1-\lambda)\,\mathcal{L}_{\text{cls}} + \lambda\,\mathcal{L}_{\text{DSR}}, \quad \mathcal{L}_{\text{DSR}} = \beta\,\mathcal{L}_{\text{tag}} + (1-\beta)\,\mathcal{L}_{\text{rat}}$$
with $\lambda = 0.3$, $\beta = 0.6$, ArcFace $(m, s) = (0.5, 64)$.


## Kaggle setup

**Before running:**
1. Create a new Kaggle notebook, set **Accelerator = GPU T4 x1**.
2. Add these datasets via the *Add data* panel (slugs may need updating to what's available on your account):
   - `parthplc/facebook-hateful-meme-dataset` → HMC
   - HarMeme: upload your own (from [MOMENTA repo](https://github.com/LCS2-IIITD/MOMENTA-A-Multimodal-Framework-for-Detecting-Harmful-Memes-and-Their-Targets))
   - PrideMM: upload your own (from the MemeCLIP project release)
3. Adjust the paths in the `CFG` cell below to match whatever your Kaggle dataset folders end up named.
4. (Optional) Pre-cache LLaVA-1.5-7B by adding `llava-hf/llava-1.5-7b-hf` as a Kaggle model dataset to avoid re-download.

**Runtime expectation (one-off Stage 1 on full HMC, T4):**
- CLIP feature extraction: ~5 minutes
- LLaVA keyword generation: ~3–5 hours (this is the slow step)
- LLaVA rationale generation: ~3–5 hours

Save the cache directory `/kaggle/working/cache` as a Kaggle dataset after Stage 1 so future Stage 4 runs (~20 min each) don't repeat the LLaVA pass.

**Stages are skippable.** Every step writes to disk and skips re-execution if its cache exists, so you can run the notebook in pieces across multiple Kaggle sessions.


In [56]:
# --- Install dependencies (one-time per session) ---
# Do NOT version-pin or upgrade transformers/tokenizers — Kaggle's base image
# has a consistent install; partial upgrades cause broken mixed states.
import subprocess, sys
from pathlib import Path

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip("sentencepiece")
_pip("ftfy", "regex")
_pip("git+https://github.com/openai/CLIP.git")

# Guard file prevents infinite restart loop: written to disk BEFORE the restart
# so the next boot sees it and skips the shutdown call.
_bnb_flag = Path("/kaggle/working/.bnb_installed")

if _bnb_flag.exists():
    import bitsandbytes as _bnb
    print(f"bitsandbytes {_bnb.__version__} ready (post-install boot)")
else:
    try:
        import bitsandbytes as _bnb
        _ver = tuple(int(x) for x in _bnb.__version__.split(".")[:2])
        if _ver < (0, 46):
            raise ImportError("too old")
        _bnb_flag.touch()
        print(f"bitsandbytes {_bnb.__version__} already present, no restart needed.")
    except Exception:
        print("Installing bitsandbytes — kernel will restart once...")
        _pip("-U", "bitsandbytes")
        _bnb_flag.touch()   # write flag BEFORE shutdown so next boot skips this branch
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)


bitsandbytes 0.49.2 ready (post-install boot)


In [57]:
# --- Imports ---
import os, sys, json, math, time, random, gc, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# Patch Pillow _typing for Kaggle base-image version skew (PIL._typing missing _Ink)
import PIL._typing as _pil_t
if not hasattr(_pil_t, "_Ink"):
    _pil_t._Ink = object  # placeholder; only the name needs to exist for ImageDraw import

from PIL import Image

import clip  # OpenAI CLIP
from transformers import (
    AutoTokenizer, BitsAndBytesConfig,
    LlavaForConditionalGeneration, LlavaProcessor,
    LlamaTokenizer, CLIPImageProcessor,
)
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm.auto import tqdm

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")


PyTorch 2.10.0+cu128  |  CUDA available: True
GPU: Tesla T4  |  VRAM: 15.6 GB


In [58]:
def set_seed(seed: int):
    """Reproducibility helper. Paper uses seeds {42, 123, 456}."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## Configuration

All hyperparameters from Section IV.C of the paper are pinned here. Edit `HMC_DIR / HARMEME_DIR / PRIDEMM_DIR` to match the actual mount points of your Kaggle datasets.


In [59]:
@dataclass
class CFG:
    # ---- Dataset paths (EDIT to match your Kaggle dataset mounts) ----
    INPUT_ROOT:   Path = Path("/kaggle/input")
    OUTPUT_ROOT:  Path = Path("/kaggle/working")
    HMC_DIR:      Path = Path("/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data")
    HARMEME_DIR:  Path = Path("/kaggle/input/harmeme")
    PRIDEMM_DIR:  Path = Path("/kaggle/input/pridemm")

    # ---- Architecture ----
    D_S:          int = 512    # CLIP ViT-B/32 embedding dim
    NUM_KEYWORDS: int = 10     # K=10 visual keywords / meme
    NUM_HEADS:    int = 8
    NUM_CLASSES:  int = 2
    DROPOUT:    float = 0.1

    # ---- Optimisation (paper Section IV.C) ----
    BATCH_SIZE:    int = 64
    LR:          float = 3e-4
    WEIGHT_DECAY:float = 1e-4
    EPOCHS_ATIN:   int = 20
    EPOCHS_RDM:    int = 30
    EPOCHS_GRACE:  int = 25

    # ---- ArcFace (Eq. 13) ----
    ARC_M: float = 0.5
    ARC_S: float = 64.0

    # ---- Loss weights (Eqs. 17, 18) ----
    LAMBDA: float = 0.3
    BETA:   float = 0.6

    # ---- Contrastive temperature (Eq. 5) ----
    TAU: float = 0.07

    # ---- Multi-seed ensemble (paper Section IV.C) ----
    SEEDS: tuple = (42, 123, 456)

    # ---- Encoders ----
    CLIP_NAME:   str = "ViT-B/32"
    LLAVA_MODEL: str = "llava-hf/llava-1.5-7b-hf"

    # ---- Generation ----
    KEYWORD_MAX_NEW_TOKENS:   int = 80
    RATIONALE_MAX_NEW_TOKENS: int = 96

    # ---- Misc ----
    NUM_WORKERS: int = 2
    DEVICE:      str = "cuda" if torch.cuda.is_available() else "cpu"
    # Set this for a tiny smoke test (e.g. 50) before launching the real run.
    DEBUG_LIMIT: Optional[int] = None

cfg = CFG()
# Use a separate cache dir for debug runs so smoke-test caches don't pollute the full run.
_cache_tag = "cache_debug" if cfg.DEBUG_LIMIT else "cache"
_ckpt_tag  = "ckpt_debug"  if cfg.DEBUG_LIMIT else "ckpt"
cfg.CACHE_ROOT = cfg.OUTPUT_ROOT / _cache_tag
cfg.CKPT_ROOT  = cfg.OUTPUT_ROOT / _ckpt_tag
cfg.CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cfg.CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Cache root: {cfg.CACHE_ROOT}  |  Ckpt root: {cfg.CKPT_ROOT}")


Cache root: /kaggle/working/cache  |  Ckpt root: /kaggle/working/ckpt


## 1. Dataset loaders

Each loader returns a `List[Dict]` of records with fields `{id, image_path, text, label}`. The three loaders are isolated so adding a new benchmark is just one more function.


In [60]:
def _load_hmc(split: str) -> List[Dict]:
    """Facebook Hateful Memes Challenge. Splits: 'train', 'dev', 'test'."""
    fp = cfg.HMC_DIR / f"{split}.jsonl"
    assert fp.exists(), (
        f"Missing {fp}. Attach the HMC dataset (e.g. 'facebook-hateful-meme-dataset') "
        f"and update CFG.HMC_DIR."
    )
    records = []
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            records.append({
                "id": str(obj["id"]),
                "image_path": str(cfg.HMC_DIR / obj["img"]),
                "text": obj.get("text", ""),
                "label": int(obj.get("label", -1)),
            })
    return records


def _load_harmeme(split: str) -> List[Dict]:
    """HarMeme (Harm-C / COVID-19). Splits: 'train', 'val', 'test'.

    The MOMENTA release uses JSONL with fields `id, image, labels, text` where
    labels is either a string or a list whose first entry is 'not harmful',
    'somewhat harmful' or 'very harmful'.
    """
    name_map = {"train": "train.jsonl", "val": "val.jsonl", "test": "test.jsonl"}
    fp = cfg.HARMEME_DIR / name_map[split]
    if not fp.exists():
        alt = list(cfg.HARMEME_DIR.glob(f"*{split}*.jsonl"))
        assert alt, f"Missing HarMeme split '{split}' under {cfg.HARMEME_DIR}"
        fp = alt[0]
    img_root = cfg.HARMEME_DIR / "img"
    if not img_root.exists():
        img_root = cfg.HARMEME_DIR / "images"
    if not img_root.exists():
        img_root = cfg.HARMEME_DIR
    records = []
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            lbl = obj.get("labels", obj.get("label", "not harmful"))
            if isinstance(lbl, list):
                lbl = lbl[0] if lbl else "not harmful"
            label = 0 if "not" in str(lbl).lower() else 1
            img = obj.get("image", obj.get("img", ""))
            records.append({
                "id": str(obj.get("id", img)),
                "image_path": str(img_root / img),
                "text": obj.get("text", obj.get("tweet_text", "")),
                "label": label,
            })
    return records


def _load_pridemm(split: str) -> List[Dict]:
    """PrideMM (Hate task). Splits via 85/5/10. Expects a CSV plus image directory."""
    csv_candidates = list(cfg.PRIDEMM_DIR.glob("*.csv"))
    assert csv_candidates, f"No CSV found under {cfg.PRIDEMM_DIR}"
    df = pd.read_csv(csv_candidates[0])
    if "split" in df.columns:
        df = df[df["split"].str.lower() == split].reset_index(drop=True)
    else:
        rng = np.random.RandomState(42)
        idx = rng.permutation(len(df))
        n_tr, n_va = int(0.85 * len(df)), int(0.05 * len(df))
        keep = {"train": idx[:n_tr], "val": idx[n_tr:n_tr + n_va], "test": idx[n_tr + n_va:]}[split]
        df = df.iloc[keep].reset_index(drop=True)
    for sub in ("Images", "images", "img"):
        if (cfg.PRIDEMM_DIR / sub).exists():
            img_root = cfg.PRIDEMM_DIR / sub
            break
    else:
        img_root = cfg.PRIDEMM_DIR
    text_col  = next((c for c in ("text", "ocr_text", "caption") if c in df.columns), df.columns[0])
    img_col   = next((c for c in ("image", "img_name", "filename") if c in df.columns), df.columns[0])
    label_col = next((c for c in ("hate", "label", "Hate") if c in df.columns), None)
    assert label_col is not None, "Could not find a hate-label column in PrideMM CSV"
    records = [{
        "id":         str(r[img_col]),
        "image_path": str(img_root / str(r[img_col])),
        "text":       str(r[text_col]),
        "label":      int(r[label_col]),
    } for _, r in df.iterrows()]
    return records


DATASET_LOADERS = {
    "hmc":     {"train": lambda: _load_hmc("train"),
                "val":   lambda: _load_hmc("dev"),
                "test":  lambda: _load_hmc("test")},
    "harmeme": {"train": lambda: _load_harmeme("train"),
                "val":   lambda: _load_harmeme("val"),
                "test":  lambda: _load_harmeme("test")},
    "pridemm": {"train": lambda: _load_pridemm("train"),
                "val":   lambda: _load_pridemm("val"),
                "test":  lambda: _load_pridemm("test")},
}

def load_split(dataset: str, split: str) -> List[Dict]:
    records = DATASET_LOADERS[dataset][split]()
    if cfg.DEBUG_LIMIT is not None:
        records = records[: cfg.DEBUG_LIMIT]
    return records


In [61]:
class MemeDataset(Dataset):
    """Stage-4 dataset: precomputed embeddings only. No image / text I/O."""
    def __init__(self, ids, x_img, x_txt, z_tag, d, y):
        assert x_img.shape == x_txt.shape == z_tag.shape == d.shape
        assert x_img.size(0) == y.size(0) == len(ids)
        self.ids   = ids
        self.x_img = x_img
        self.x_txt = x_txt
        self.z_tag = z_tag
        self.d     = d
        self.y     = y

    def __len__(self):
        return self.y.size(0)

    def __getitem__(self, i):
        return {
            "x_img": self.x_img[i],
            "x_txt": self.x_txt[i],
            "z_tag": self.z_tag[i],
            "d":     self.d[i],
            "y":     self.y[i],
        }


## 2. Stage 1a — CLIP feature extraction (Eq. 1)

For every meme we cache:
- `x_img.pt`  shape `[N, 512]`  — frozen CLIP ViT-B/32 image embedding
- `x_txt.pt`  shape `[N, 512]`  — frozen CLIP text embedding of the overlaid text
- `y.pt`      shape `[N]`        — binary label
- `ids.json / text.json / image_paths.json` — bookkeeping for later stages


In [62]:
def precompute_clip(dataset: str, split: str, force: bool = False):
    cache_dir = cfg.CACHE_ROOT / dataset / split
    cache_dir.mkdir(parents=True, exist_ok=True)
    fp_img = cache_dir / "x_img.pt"
    fp_txt = cache_dir / "x_txt.pt"
    fp_lbl = cache_dir / "y.pt"
    fp_ids = cache_dir / "ids.json"
    fp_text = cache_dir / "text.json"
    fp_paths = cache_dir / "image_paths.json"
    targets = [fp_img, fp_txt, fp_lbl, fp_ids, fp_text, fp_paths]

    if not force and all(p.exists() for p in targets):
        print(f"[{dataset}/{split}] CLIP cache present, skipping.")
        return

    records = load_split(dataset, split)
    print(f"[{dataset}/{split}] {len(records)} samples")

    model, preprocess = clip.load(cfg.CLIP_NAME, device=cfg.DEVICE)
    model.eval()

    ids, texts, paths, labels = [], [], [], []
    img_chunks, txt_chunks = [], []

    with torch.no_grad():
        for r in tqdm(records, desc=f"CLIP {dataset}/{split}"):
            try:
                img = preprocess(Image.open(r["image_path"]).convert("RGB")) \
                    .unsqueeze(0).to(cfg.DEVICE)
                tok = clip.tokenize(r["text"][:300] or " ", truncate=True).to(cfg.DEVICE)
            except Exception as e:
                print(f"  skip {r['id']}: {e}")
                continue
            img_chunks.append(model.encode_image(img).cpu().float())
            txt_chunks.append(model.encode_text(tok).cpu().float())
            ids.append(r["id"])
            texts.append(r["text"])
            paths.append(r["image_path"])
            labels.append(r["label"])

    x_img = torch.cat(img_chunks, dim=0)
    x_txt = torch.cat(txt_chunks, dim=0)
    y = torch.tensor(labels, dtype=torch.long)

    torch.save(x_img, fp_img)
    torch.save(x_txt, fp_txt)
    torch.save(y, fp_lbl)
    fp_ids.write_text(json.dumps(ids))
    fp_text.write_text(json.dumps(texts))
    fp_paths.write_text(json.dumps(paths))

    print(f"[{dataset}/{split}] saved x_img {tuple(x_img.shape)}  x_txt {tuple(x_txt.shape)}")
    del model
    gc.collect()
    torch.cuda.empty_cache()


## 3. Stage 1b — LLaVA keyword generation

For each meme image, LLaVA-1.5-7B (4-bit) emits ten single-word visual concepts. The prompt is deliberately bias-free per [MemeTAG, 2026] — it asks only for *visible* content, no sentiment or interpretation, to keep the keywords descriptive.

This is the slowest cell in the pipeline; it checkpoints to disk every 250 memes so you can resume after a Kaggle session timeout.


In [63]:
def load_llava():
    """Load LLaVA-1.5-7B with 4-bit NF4 quantization (~5 GB VRAM)."""
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    # Bypass AutoProcessor entirely: LlamaTokenizer uses SentencePiece (no Rust tokenizers
    # library needed), and CLIPImageProcessor has no version issues.
    tokenizer = LlamaTokenizer.from_pretrained(cfg.LLAVA_MODEL, use_fast=False)
    if not tokenizer.model_max_length or tokenizer.model_max_length > 1_000_000:
        tokenizer.model_max_length = 2048  # LlavaProcessor needs this for internal padding
    image_proc = CLIPImageProcessor.from_pretrained(cfg.LLAVA_MODEL)
    # Do NOT pass patch_size/vision_feature_select_strategy: their interaction with
    # the installed transformers version causes either a // NoneType error (patch_size=None)
    # or an off-by-one mismatch (patch_size=14 → 575 tokens vs 576 model features).
    # We instead pre-expand <image> manually in every prompt (see _LLAVA_IMG_TOKENS).
    processor = LlavaProcessor(tokenizer=tokenizer, image_processor=image_proc)
    model = LlavaForConditionalGeneration.from_pretrained(
        cfg.LLAVA_MODEL,
        quantization_config=bnb,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    return model, processor


def _llava_inputs(model, processor, img: Image.Image, text: str) -> dict:
    """Prepare inputs without calling LlavaProcessor.__call__.

    LlavaProcessor has version-specific bugs: it does image_size // patch_size
    with patch_size=None, raising TypeError even after pre-expansion. We bypass it:
    call tokenizer and image_processor separately and read patch_size from model.config.
    """
    ps  = model.config.vision_config.patch_size   # 14 for LLaVA-1.5-7B ViT-L/14
    sz  = model.config.vision_config.image_size   # 336
    n   = (sz // ps) ** 2                         # 576 patch tokens (CLS excluded by model)

    expanded = text.replace("<image>", "<image>" * n)

    pv  = processor.image_processor(images=img, return_tensors="pt").pixel_values
    tok = processor.tokenizer(
        expanded, return_tensors="pt", truncation=True,
        max_length=processor.tokenizer.model_max_length or 2048,
    )
    dev = next(model.parameters()).device
    return {
        "input_ids":      tok.input_ids.to(dev),
        "attention_mask": tok.attention_mask.to(dev),
        "pixel_values":   pv.to(dev, torch.float16),
    }


KEYWORD_PROMPT = (
    "USER: <image>\nList exactly TEN single-word visual concepts that describe "
    "ONLY what is objectively visible in this image (e.g. people, objects, colors, "
    "settings, expressions). Do not interpret intent, do not judge content, and do "
    "not use sentiment words. Output one line of ten comma-separated words.\nASSISTANT:"
)


def _parse_keywords(text: str, k: int = 10) -> List[str]:
    text = text.split("ASSISTANT:")[-1].strip().split("\n")[0]
    parts = re.split(r"[,;]", text)
    words = []
    for p in parts:
        w = re.sub(r"[^A-Za-z\-']+", " ", p).strip().split()
        if w:
            words.append(w[0].lower())
        if len(words) >= k:
            break
    while len(words) < k:
        words.append("unknown")
    return words[:k]


def precompute_keywords(dataset: str, split: str, force: bool = False):
    cache_dir = cfg.CACHE_ROOT / dataset / split
    fp = cache_dir / "keywords.json"
    if not force and fp.exists():
        existing = json.loads(fp.read_text())
        ids_full = json.loads((cache_dir / "ids.json").read_text())
        if all(i in existing for i in ids_full):
            print(f"[{dataset}/{split}] keyword cache complete, skipping.")
            return

    image_paths = json.loads((cache_dir / "image_paths.json").read_text())
    ids = json.loads((cache_dir / "ids.json").read_text())

    out: Dict[str, List[str]] = {}
    if fp.exists():
        out = json.loads(fp.read_text())

    print(f"[{dataset}/{split}] generating keywords for {len(ids) - len(out)} new images")
    model, processor = load_llava()

    with torch.inference_mode():
        for i, (id_, path) in enumerate(tqdm(list(zip(ids, image_paths)),
                                             desc=f"Keywords {dataset}/{split}")):
            if id_ in out:
                continue
            try:
                img = Image.open(path).convert("RGB")
                inputs = _llava_inputs(model, processor, img, KEYWORD_PROMPT)
                gen = model.generate(
                    **inputs,
                    max_new_tokens=cfg.KEYWORD_MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id,
                )
                raw = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
                out[id_] = _parse_keywords(raw, cfg.NUM_KEYWORDS)
            except Exception as e:
                print(f"  fail {id_}: {e}")
                out[id_] = ["unknown"] * cfg.NUM_KEYWORDS
            if (i + 1) % 250 == 0:
                fp.write_text(json.dumps(out))

    fp.write_text(json.dumps(out))
    print(f"[{dataset}/{split}] saved {len(out)} keyword lists -> {fp}")
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()


## 4. Stage 1c — CLIP-text-encode the keywords

The ten keyword strings per meme are encoded by CLIP's text tower. Output: `E_keywords.pt` of shape `[N, 10, 512]`.


In [64]:
def encode_keywords(dataset: str, split: str, force: bool = False):
    cache_dir = cfg.CACHE_ROOT / dataset / split
    fp = cache_dir / "E_keywords.pt"
    if not force and fp.exists():
        print(f"[{dataset}/{split}] E_keywords cache present.")
        return

    keywords = json.loads((cache_dir / "keywords.json").read_text())
    ids = json.loads((cache_dir / "ids.json").read_text())

    model, _ = clip.load(cfg.CLIP_NAME, device=cfg.DEVICE)
    model.eval()

    chunks = []
    with torch.no_grad():
        for id_ in tqdm(ids, desc=f"Encode keywords {dataset}/{split}"):
            kws = keywords.get(id_, ["unknown"] * cfg.NUM_KEYWORDS)
            tok = clip.tokenize(kws, truncate=True).to(cfg.DEVICE)
            chunks.append(model.encode_text(tok).cpu().float().unsqueeze(0))

    E = torch.cat(chunks, dim=0)  # [N, 10, 512]
    torch.save(E, fp)
    print(f"[{dataset}/{split}] saved E_keywords {tuple(E.shape)}")
    del model
    gc.collect()
    torch.cuda.empty_cache()


## 5. Stage 2 — ATIN (Aggregated Tag Inference Network, Eqs. 2–3)

A multi-head self-attention module with a learnable query `q`. We train it independently with a simple linear hate-classification head so the keyword aggregation learns a task-aware projection; afterwards the ATIN backbone is frozen and used to produce `z_tag` for every split.


In [65]:
class ATIN(nn.Module):
    """MHSA with a learnable query.

    Implements Eqs. 2-3:
        E' = LN(E + MHSA(q, E, E))
        z_tag = mean_i e'_i  in R^{d_s}
    """
    def __init__(self, d_s: int = 512, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.q = nn.Parameter(torch.randn(1, 1, d_s) * 0.02)
        self.mha = nn.MultiheadAttention(d_s, num_heads,
                                         batch_first=True, dropout=dropout)
        self.ln = nn.LayerNorm(d_s)

    def forward(self, E: torch.Tensor) -> torch.Tensor:
        # E: [B, K, d_s]
        B, K, _ = E.shape
        q = self.q.expand(B, K, -1)
        attn, _ = self.mha(q, E, E)
        E_prime = self.ln(E + attn)
        return E_prime.mean(dim=1)  # [B, d_s]


class ATINClassifier(nn.Module):
    """ATIN + classification head used only during Stage 2 training."""
    def __init__(self, d_s=512, num_heads=8, num_classes=2, dropout=0.1):
        super().__init__()
        self.atin = ATIN(d_s, num_heads, dropout)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_s, d_s // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_s // 2, num_classes),
        )

    def forward(self, E):
        z_tag = self.atin(E)
        return self.head(z_tag), z_tag


def train_atin(dataset: str, force: bool = False):
    fp_ckpt = cfg.CKPT_ROOT / f"atin_{dataset}.pt"
    if not force and fp_ckpt.exists():
        print(f"[{dataset}] ATIN ckpt present -> {fp_ckpt}")
        return

    cache_tr = cfg.CACHE_ROOT / dataset / "train"
    cache_va = cfg.CACHE_ROOT / dataset / "val"
    E_tr = torch.load(cache_tr / "E_keywords.pt")
    y_tr = torch.load(cache_tr / "y.pt")
    E_va = torch.load(cache_va / "E_keywords.pt")
    y_va = torch.load(cache_va / "y.pt")

    set_seed(42)
    model = ATINClassifier(cfg.D_S, cfg.NUM_HEADS, cfg.NUM_CLASSES,
                           cfg.DROPOUT).to(cfg.DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()

    dl_tr = DataLoader(TensorDataset(E_tr, y_tr),
                       batch_size=cfg.BATCH_SIZE, shuffle=True)
    dl_va = DataLoader(TensorDataset(E_va, y_va),
                       batch_size=cfg.BATCH_SIZE, shuffle=False)

    best_val, best_state = -1.0, None
    for epoch in range(cfg.EPOCHS_ATIN):
        model.train()
        run = 0.0; n = 0
        for E_b, y_b in dl_tr:
            E_b, y_b = E_b.to(cfg.DEVICE), y_b.to(cfg.DEVICE)
            logits, _ = model(E_b)
            loss = crit(logits, y_b)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * y_b.size(0); n += y_b.size(0)
        # quick val
        model.eval()
        with torch.no_grad():
            logits_va = torch.cat([model(eb.to(cfg.DEVICE))[0].cpu()
                                   for eb, _ in dl_va])
        val_acc = (logits_va.argmax(1) == y_va).float().mean().item()
        print(f"  ATIN[{dataset}] epoch {epoch+1}/{cfg.EPOCHS_ATIN}  "
              f"loss={run/n:.4f}  val_acc={val_acc:.4f}")
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save({"atin": model.atin.state_dict()}, fp_ckpt)
    print(f"[{dataset}] best val_acc={best_val:.4f}  saved {fp_ckpt}")


def precompute_z_tag(dataset: str, force: bool = False):
    """Run frozen ATIN to produce z_tag for train/val/test."""
    fp_ckpt = cfg.CKPT_ROOT / f"atin_{dataset}.pt"
    assert fp_ckpt.exists(), f"Train ATIN first ({fp_ckpt} missing)"
    atin = ATIN(cfg.D_S, cfg.NUM_HEADS, cfg.DROPOUT).to(cfg.DEVICE)
    atin.load_state_dict(torch.load(fp_ckpt, map_location=cfg.DEVICE)["atin"])
    atin.eval()

    for split in ("train", "val", "test"):
        cache_dir = cfg.CACHE_ROOT / dataset / split
        fp = cache_dir / "z_tag.pt"
        if not force and fp.exists():
            print(f"[{dataset}/{split}] z_tag present.")
            continue
        E = torch.load(cache_dir / "E_keywords.pt")
        chunks = []
        with torch.no_grad():
            for i in range(0, E.size(0), 256):
                chunks.append(atin(E[i:i + 256].to(cfg.DEVICE)).cpu())
        z_tag = torch.cat(chunks, dim=0)
        torch.save(z_tag, fp)
        print(f"[{dataset}/{split}] saved z_tag {tuple(z_tag.shape)}")


## 6. Stage 1d/e — LLaVA rationale + CLIP encoding

A single concise rationale sentence per meme grounded in BOTH the image and the overlaid text. Paper uses GPT-4o or LLaVA-13B; we use LLaVA-1.5-7B (already in memory the first time, then reload) so the whole pipeline stays free-tier.


In [66]:
RATIONALE_PROMPT_TMPL = (
    "USER: <image>\nThis meme is overlaid with the text: \"{text}\". "
    "In ONE concise sentence (under 25 words), describe the cultural reference, "
    "stereotype, joke structure, or implicit meaning that a viewer needs to "
    "understand the meme's intent. Output exactly one sentence.\nASSISTANT:"
)


def _parse_rationale(text: str) -> str:
    text = text.split("ASSISTANT:")[-1].strip()
    text = re.split(r"(?<=[.!?])\s", text)[0].strip()
    return text[:400] if text else "A meme combining an image and overlaid text."


def precompute_rationales(dataset: str, split: str, force: bool = False):
    cache_dir = cfg.CACHE_ROOT / dataset / split
    fp = cache_dir / "rationales.json"
    if not force and fp.exists():
        existing = json.loads(fp.read_text())
        ids_full = json.loads((cache_dir / "ids.json").read_text())
        if all(i in existing for i in ids_full):
            print(f"[{dataset}/{split}] rationale cache complete, skipping.")
            return

    ids   = json.loads((cache_dir / "ids.json").read_text())
    paths = json.loads((cache_dir / "image_paths.json").read_text())
    texts = json.loads((cache_dir / "text.json").read_text())

    out: Dict[str, str] = {}
    if fp.exists():
        out = json.loads(fp.read_text())

    print(f"[{dataset}/{split}] generating rationales for {len(ids) - len(out)} memes")
    model, processor = load_llava()

    with torch.inference_mode():
        for i, (id_, path, text) in enumerate(tqdm(list(zip(ids, paths, texts)),
                                                   desc=f"Rationales {dataset}/{split}")):
            if id_ in out:
                continue
            try:
                img = Image.open(path).convert("RGB")
                prompt = RATIONALE_PROMPT_TMPL.format(text=(text or "")[:200].replace('"', "'"))
                inputs = _llava_inputs(model, processor, img, prompt)
                gen = model.generate(
                    **inputs,
                    max_new_tokens=cfg.RATIONALE_MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id,
                )
                raw = processor.tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
                out[id_] = _parse_rationale(raw)
            except Exception as e:
                print(f"  fail {id_}: {e}")
                out[id_] = "A meme combining an image and overlaid text."
            if (i + 1) % 250 == 0:
                fp.write_text(json.dumps(out))

    fp.write_text(json.dumps(out))
    print(f"[{dataset}/{split}] saved {len(out)} rationales -> {fp}")
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()


def encode_rationales(dataset: str, split: str, force: bool = False):
    cache_dir = cfg.CACHE_ROOT / dataset / split
    fp = cache_dir / "v_rationale.pt"
    if not force and fp.exists():
        print(f"[{dataset}/{split}] v_rationale cache present.")
        return

    rationales = json.loads((cache_dir / "rationales.json").read_text())
    ids = json.loads((cache_dir / "ids.json").read_text())

    model, _ = clip.load(cfg.CLIP_NAME, device=cfg.DEVICE)
    model.eval()

    chunks = []
    with torch.no_grad():
        for i in range(0, len(ids), 128):
            batch_ids = ids[i:i + 128]
            batch_txt = [rationales.get(j, "A meme.") for j in batch_ids]
            tok = clip.tokenize(batch_txt, truncate=True).to(cfg.DEVICE)
            chunks.append(model.encode_text(tok).cpu().float())
    v = torch.cat(chunks, dim=0)
    torch.save(v, fp)
    print(f"[{dataset}/{split}] saved v_rationale {tuple(v.shape)}")
    del model
    gc.collect()
    torch.cuda.empty_cache()


## 7. Stage 3 — RDM (Rationale Distillation Module, Eqs. 4–5)

A 2-layer MLP that compresses CLIP-encoded rationales `v` into compact guidance vectors `d`. Trained with NT-Xent contrastive loss: positives are `(d_n, z_tag,n)` from the same meme; negatives are cross-meme pairs in the same batch.

We use the symmetric InfoNCE form (mean of `d→z_tag` and `z_tag→d`) since both directions provide gradient signal at no extra compute.


In [67]:
class RDM(nn.Module):
    """Rationale Distillation Module: d = W2 ReLU(W1 v + b1) + b2 (Eq. 4)."""
    def __init__(self, d_s: int = 512, d_hidden: int = 512, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_s, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_s),
        )

    def forward(self, v):
        return self.net(v)


def nt_xent_loss(d, z_tag, tau: float = 0.07):
    """Symmetric NT-Xent / InfoNCE (Eq. 5)."""
    d_n = F.normalize(d, dim=1)
    z_n = F.normalize(z_tag, dim=1)
    sim = d_n @ z_n.t() / tau         # [B, B] cosine similarities scaled by 1/tau
    labels = torch.arange(d.size(0), device=d.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))


def train_rdm(dataset: str, force: bool = False):
    fp_ckpt = cfg.CKPT_ROOT / f"rdm_{dataset}.pt"
    if not force and fp_ckpt.exists():
        print(f"[{dataset}] RDM ckpt present -> {fp_ckpt}")
        return

    cache_tr = cfg.CACHE_ROOT / dataset / "train"
    v_tr = torch.load(cache_tr / "v_rationale.pt")
    z_tr = torch.load(cache_tr / "z_tag.pt")

    set_seed(42)
    rdm = RDM(cfg.D_S).to(cfg.DEVICE)
    opt = torch.optim.AdamW(rdm.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY)

    dl = DataLoader(TensorDataset(v_tr, z_tr),
                    batch_size=min(cfg.BATCH_SIZE, len(v_tr)), shuffle=True, drop_last=False)

    rdm.train()
    for epoch in range(cfg.EPOCHS_RDM):
        run = 0.0; n = 0
        for v_b, z_b in dl:
            v_b, z_b = v_b.to(cfg.DEVICE), z_b.to(cfg.DEVICE)
            d_b = rdm(v_b)
            loss = nt_xent_loss(d_b, z_b, cfg.TAU)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * v_b.size(0); n += v_b.size(0)
        print(f"  RDM[{dataset}] epoch {epoch+1}/{cfg.EPOCHS_RDM}  loss={run/n:.4f}")

    torch.save({"rdm": rdm.state_dict()}, fp_ckpt)
    print(f"[{dataset}] saved {fp_ckpt}")


def precompute_d(dataset: str, force: bool = False):
    fp_ckpt = cfg.CKPT_ROOT / f"rdm_{dataset}.pt"
    rdm = RDM(cfg.D_S).to(cfg.DEVICE)
    rdm.load_state_dict(torch.load(fp_ckpt, map_location=cfg.DEVICE)["rdm"])
    rdm.eval()
    for split in ("train", "val", "test"):
        cache_dir = cfg.CACHE_ROOT / dataset / split
        fp = cache_dir / "d.pt"
        if not force and fp.exists():
            print(f"[{dataset}/{split}] d present.")
            continue
        v = torch.load(cache_dir / "v_rationale.pt")
        chunks = []
        with torch.no_grad():
            for i in range(0, v.size(0), 256):
                chunks.append(rdm(v[i:i + 256].to(cfg.DEVICE)).cpu())
        d = torch.cat(chunks, dim=0)
        torch.save(d, fp)
        print(f"[{dataset}/{split}] saved d {tuple(d.shape)}")


## 8. Stage 4 — GRACE model

### 8.1 Modality adapters (Eqs. 6–7)
$$\mathbf{x}'_{\text{img}} = \text{LinearProj}_V(\mathbf{x}_{\text{img}}), \quad \mathbf{z}^*_{\text{img}} = \alpha_V f_V(\mathbf{x}'_{\text{img}}) + (1 - \alpha_V)\mathbf{x}'_{\text{img}}$$

`alpha` is parameterised in raw form and squashed through sigmoid so it stays in `[0, 1]` without constraints.

### 8.2 DPCAF (Eqs. 8–12)
Two asymmetric streams:
$$\mathbf{A}_{I\to T} = \text{softmax}\!\left(\frac{\mathbf{z}^*_{\text{img}}(\mathbf{z}^*_{\text{txt}})^\top}{\sqrt{d_s}}\right)\mathbf{z}^*_{\text{txt}}, \quad \mathbf{A}_{T\to I} = \text{softmax}\!\left(\frac{\mathbf{z}^*_{\text{txt}}(\mathbf{z}^*_{\text{img}})^\top}{\sqrt{d_s}}\right)\mathbf{z}^*_{\text{img}}$$
implemented as **feature-level outer-product attention** (the same interaction-matrix style HateCLIPper uses), preserving directional dependency that an element-wise product collapses.

### 8.3 ArcFace head (Eq. 13)
$$\ell_j = \begin{cases} s\cos(\theta_j + m) & j = y \\ s\cos(\theta_j) & \text{else} \end{cases}, \quad m = 0.5, s = 64$$

At inference no margin is applied (we want pure cosine ranking for AUROC).

### 8.4 DSR loss (Eqs. 14–17)
Two MLPs reconstruct `z_tag` and `d` from the fused feature; the DSR loss is the convex combination of their cosine distances with `beta=0.6`.


In [68]:
class ModalityAdapter(nn.Module):
    """Linear projection + 2-layer MLP adapter, residually mixed via learnable alpha (Eqs. 6-7)."""
    def __init__(self, d_s: int = 512, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(d_s, d_s)
        self.adapter = nn.Sequential(
            nn.Linear(d_s, d_s),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_s, d_s),
        )
        # Sigmoid keeps alpha in [0, 1]; init -> 0.5
        self.alpha_raw = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xp = self.proj(x)
        a = torch.sigmoid(self.alpha_raw)
        return a * self.adapter(xp) + (1 - a) * xp


class DPCAF(nn.Module):
    """Dual-Path Cross-Attention Fusion (Eqs. 8-12)."""
    def __init__(self, d_s: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_s = d_s
        self.scale = 1.0 / math.sqrt(d_s)
        self.proj_fusion = nn.Linear(2 * d_s, d_s)
        self.ln = nn.LayerNorm(d_s)
        self.gate = nn.Linear(2 * d_s, d_s)
        self.drop = nn.Dropout(dropout)

    def _cross(self, q: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        """Feature-level cross-attention: softmax(q kv^T / sqrt(d_s)) @ kv.

        q, kv:  [B, d_s]
        Returns: [B, d_s]
        """
        # Outer product -> [B, d_s, d_s]
        A = torch.bmm(q.unsqueeze(-1), kv.unsqueeze(-2)) * self.scale
        A = F.softmax(A, dim=-1)
        # Weighted sum back to [B, d_s]
        return torch.bmm(A, kv.unsqueeze(-1)).squeeze(-1)

    def forward(self, z_img: torch.Tensor, z_txt: torch.Tensor) -> torch.Tensor:
        a_i2t = self._cross(z_img, z_txt)        # Eq. 8
        a_t2i = self._cross(z_txt, z_img)        # Eq. 9
        f_dual = self.ln(self.proj_fusion(torch.cat([a_i2t, a_t2i], dim=-1)))  # Eq. 10
        g = torch.sigmoid(self.gate(torch.cat([z_img, z_txt], dim=-1)))        # Eq. 11
        residual = 0.5 * (F.normalize(z_img, dim=-1) + F.normalize(z_txt, dim=-1))
        f = g * f_dual + (1 - g) * residual                                    # Eq. 12
        return self.drop(f)


class ArcFaceHead(nn.Module):
    """Additive-angular-margin head (Deng et al. 2019). m=0.5, s=64 from paper."""
    def __init__(self, d_in: int, num_classes: int, m: float = 0.5, s: float = 64.0):
        super().__init__()
        self.W = nn.Parameter(torch.empty(num_classes, d_in))
        nn.init.xavier_uniform_(self.W)
        self.m = m
        self.s = s

    def forward(self, x: torch.Tensor, y: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Upcast for arccos numerical stability
        Wn = F.normalize(self.W.float(), dim=1)
        xn = F.normalize(x.float(), dim=1)
        cos = (xn @ Wn.t()).clamp(-1 + 1e-7, 1 - 1e-7)
        if y is None:
            return self.s * cos
        theta = torch.acos(cos)
        cos_m = torch.cos(theta + self.m)
        one_hot = F.one_hot(y, num_classes=self.W.size(0)).float()
        return self.s * (one_hot * cos_m + (1 - one_hot) * cos)


class GRACE(nn.Module):
    """Full GRACE model. Stage-4 trainable parameters only."""
    def __init__(self, d_s: int = 512, num_classes: int = 2,
                 dropout: float = 0.1, arc_m: float = 0.5, arc_s: float = 64.0):
        super().__init__()
        self.adapter_img = ModalityAdapter(d_s, dropout)
        self.adapter_txt = ModalityAdapter(d_s, dropout)
        self.dpcaf = DPCAF(d_s, dropout)
        # Pre-output layer (Section III.D)
        self.pre = nn.Sequential(
            nn.Linear(d_s, d_s),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.arcface = ArcFaceHead(d_s, num_classes, arc_m, arc_s)
        # DSR reconstruction heads (Eq. 14)
        self.mlp_tag = nn.Sequential(
            nn.Linear(d_s, d_s), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_s, d_s),
        )
        self.mlp_rat = nn.Sequential(
            nn.Linear(d_s, d_s), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_s, d_s),
        )

    def forward(self, x_img, x_txt, y=None):
        z_img = self.adapter_img(x_img)
        z_txt = self.adapter_txt(x_txt)
        f = self.dpcaf(z_img, z_txt)
        fp = self.pre(f)
        logits = self.arcface(fp, y)
        return logits, self.mlp_tag(f), self.mlp_rat(f)


def cosine_distance(a, b):
    """1 - mean cosine similarity. Used by both DSR terms."""
    return (1 - F.cosine_similarity(a, b, dim=-1)).mean()


def grace_loss(logits, z_tag_hat, d_hat, y, z_tag, d, lam=0.3, beta=0.6):
    """Eqs. 15-18."""
    L_cls = F.cross_entropy(logits, y)
    L_tag = cosine_distance(z_tag_hat, z_tag)
    L_rat = cosine_distance(d_hat, d)
    L_dsr = beta * L_tag + (1 - beta) * L_rat
    L = (1 - lam) * L_cls + lam * L_dsr
    return L, {"loss": L.item(), "cls": L_cls.item(), "tag": L_tag.item(), "rat": L_rat.item()}


## 9. Training loop & multi-seed runner

- AdamW (lr=3e-4, wd=1e-4), cosine LR schedule, grad clip 1.0
- Mixed precision (AMP) for T4 speed
- Best checkpoint selected on validation AUROC
- Multi-seed mean ± std reported per paper protocol


In [69]:
@torch.no_grad()
def evaluate(model: GRACE, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    all_logits, all_y = [], []
    for batch in loader:
        logits, _, _ = model(
            batch["x_img"].to(cfg.DEVICE),
            batch["x_txt"].to(cfg.DEVICE),
        )
        all_logits.append(logits.detach().cpu())
        all_y.append(batch["y"])
    logits = torch.cat(all_logits, dim=0)
    y = torch.cat(all_y, dim=0)
    probs = F.softmax(logits, dim=-1)[:, 1].numpy()
    preds = logits.argmax(dim=-1).numpy()
    y_np = y.numpy()
    return {
        "acc":      accuracy_score(y_np, preds),
        "auroc":    roc_auc_score(y_np, probs) if len(set(y_np)) > 1 else float("nan"),
        "macro_f1": f1_score(y_np, preds, average="macro"),
    }


def fit_grace(dataset: str, seed: int) -> Dict[str, float]:
    set_seed(seed)
    base = cfg.CACHE_ROOT / dataset

    def _make_ds(split):
        d = base / split
        return MemeDataset(
            ids   = json.loads((d / "ids.json").read_text()),
            x_img = torch.load(d / "x_img.pt"),
            x_txt = torch.load(d / "x_txt.pt"),
            z_tag = torch.load(d / "z_tag.pt"),
            d     = torch.load(d / "d.pt"),
            y     = torch.load(d / "y.pt"),
        )

    ds_tr = _make_ds("train")
    ds_va = _make_ds("val")
    ds_te = _make_ds("test")

    dl_tr = DataLoader(ds_tr, batch_size=min(cfg.BATCH_SIZE, len(ds_tr)), shuffle=True,
                       num_workers=cfg.NUM_WORKERS, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=cfg.BATCH_SIZE, shuffle=False)
    dl_te = DataLoader(ds_te, batch_size=cfg.BATCH_SIZE, shuffle=False)

    model = GRACE(cfg.D_S, cfg.NUM_CLASSES, cfg.DROPOUT,
                  cfg.ARC_M, cfg.ARC_S).to(cfg.DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS_GRACE)
    scaler = GradScaler("cuda", enabled=(cfg.DEVICE == "cuda"))

    best_auroc, best_state = -1.0, None

    for epoch in range(cfg.EPOCHS_GRACE):
        model.train()
        run = 0.0; n = 0
        for batch in dl_tr:
            x_img = batch["x_img"].to(cfg.DEVICE)
            x_txt = batch["x_txt"].to(cfg.DEVICE)
            z_tag = batch["z_tag"].to(cfg.DEVICE)
            d_emb = batch["d"].to(cfg.DEVICE)
            y     = batch["y"].to(cfg.DEVICE)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(cfg.DEVICE == "cuda")):
                logits, z_hat, d_hat = model(x_img, x_txt, y)
                loss, _ = grace_loss(logits, z_hat, d_hat, y,
                                     z_tag, d_emb, cfg.LAMBDA, cfg.BETA)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            run += loss.item() * y.size(0); n += y.size(0)

        sched.step()  # per-epoch — T_max=EPOCHS_GRACE means one full cosine cycle over all epochs
        val = evaluate(model, dl_va)
        print(f"  seed={seed}  epoch {epoch+1:>2}/{cfg.EPOCHS_GRACE}  "
              f"train_loss={run/n:.4f}  val_auroc={val['auroc']:.4f}  val_acc={val['acc']:.4f}")
        if not math.isnan(val["auroc"]) and val["auroc"] > best_auroc:
            best_auroc = val["auroc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    # HMC test.jsonl has no ground-truth labels (all -1); fall back to val set for final metrics.
    # Other datasets (HarMeme, PrideMM) have labelled test splits, so dl_te works for them.
    test_raw = evaluate(model, dl_te)
    if math.isnan(test_raw["auroc"]):
        test = evaluate(model, dl_va)
        print(f"  seed={seed}  TEST(=val)  auroc={test['auroc']:.4f}  acc={test['acc']:.4f}  f1={test['macro_f1']:.4f}  (test labels unavailable)")
    else:
        test = test_raw
        print(f"  seed={seed}  TEST  auroc={test['auroc']:.4f}  acc={test['acc']:.4f}  f1={test['macro_f1']:.4f}")
    return test


def run_multi_seed(dataset: str) -> Dict[str, Dict[str, float]]:
    results = [fit_grace(dataset, s) for s in cfg.SEEDS]
    keys = results[0].keys()
    mean = {k: float(np.mean([r[k] for r in results])) for k in keys}
    std  = {k: float(np.std ([r[k] for r in results])) for k in keys}
    print(f"\n[{dataset}] Multi-seed test (n={len(cfg.SEEDS)}):")
    for k in keys:
        print(f"  {k:8s}: {mean[k]:.4f} ± {std[k]:.4f}")
    return {"mean": mean, "std": std, "per_seed": results}


## 9.5 Ablation study (reproduces paper Table IV)

We refactor the fusion block so it can switch between three modes:

| `fusion_mode` | What it does | Paper row |
|---|---|---|
| `dpcaf` | Full DPCAF with learnable gate | GRACE (Full) |
| `dpcaf_nogate` | DPCAF streams + projection, no gate | w/o Gate in DPCAF |
| `elem_mul` | MemeTAG-style `Norm(z_img) ⊙ Norm(z_txt)` | w/o DPCAF |

Combined with `beta=1.0` (tag-only DSR) and `use_dsr=False` (cls-only), this gives every row in Table IV.


In [70]:
class FusionBlock(nn.Module):
    """Mode-switchable fusion: full DPCAF, DPCAF-no-gate, or element-wise mul."""
    def __init__(self, d_s: int = 512, dropout: float = 0.1, mode: str = "dpcaf"):
        super().__init__()
        assert mode in ("dpcaf", "dpcaf_nogate", "elem_mul"), f"unknown mode {mode}"
        self.mode = mode
        self.drop = nn.Dropout(dropout)
        if mode in ("dpcaf", "dpcaf_nogate"):
            self.scale = 1.0 / math.sqrt(d_s)
            self.proj_fusion = nn.Linear(2 * d_s, d_s)
            self.ln_dual = nn.LayerNorm(d_s)
            if mode == "dpcaf":
                self.gate = nn.Linear(2 * d_s, d_s)
        else:  # elem_mul
            self.ln_ew = nn.LayerNorm(d_s)

    def _cross(self, q, kv):
        A = torch.bmm(q.unsqueeze(-1), kv.unsqueeze(-2)) * self.scale
        A = F.softmax(A, dim=-1)
        return torch.bmm(A, kv.unsqueeze(-1)).squeeze(-1)

    def forward(self, z_img, z_txt):
        if self.mode == "elem_mul":
            f = self.ln_ew(F.normalize(z_img, dim=-1) * F.normalize(z_txt, dim=-1))
            return self.drop(f)
        a_i2t = self._cross(z_img, z_txt)
        a_t2i = self._cross(z_txt, z_img)
        f_dual = self.ln_dual(self.proj_fusion(torch.cat([a_i2t, a_t2i], dim=-1)))
        if self.mode == "dpcaf_nogate":
            return self.drop(f_dual)
        g = torch.sigmoid(self.gate(torch.cat([z_img, z_txt], dim=-1)))
        residual = 0.5 * (F.normalize(z_img, dim=-1) + F.normalize(z_txt, dim=-1))
        return self.drop(g * f_dual + (1 - g) * residual)


class GRACEAblation(nn.Module):
    """GRACE with switchable fusion. Loss-level ablations are handled in grace_loss_ablation."""
    def __init__(self, d_s=512, num_classes=2, dropout=0.1,
                 arc_m=0.5, arc_s=64.0, fusion_mode="dpcaf"):
        super().__init__()
        self.adapter_img = ModalityAdapter(d_s, dropout)
        self.adapter_txt = ModalityAdapter(d_s, dropout)
        self.fusion = FusionBlock(d_s, dropout, mode=fusion_mode)
        self.pre = nn.Sequential(nn.Linear(d_s, d_s), nn.ReLU(), nn.Dropout(dropout))
        self.arcface = ArcFaceHead(d_s, num_classes, arc_m, arc_s)
        self.mlp_tag = nn.Sequential(
            nn.Linear(d_s, d_s), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_s, d_s),
        )
        self.mlp_rat = nn.Sequential(
            nn.Linear(d_s, d_s), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_s, d_s),
        )

    def forward(self, x_img, x_txt, y=None):
        z_img = self.adapter_img(x_img)
        z_txt = self.adapter_txt(x_txt)
        f = self.fusion(z_img, z_txt)
        fp = self.pre(f)
        logits = self.arcface(fp, y)
        return logits, self.mlp_tag(f), self.mlp_rat(f)


def grace_loss_ablation(logits, z_tag_hat, d_hat, y, z_tag, d,
                        lam=0.3, beta=0.6, use_dsr=True):
    """Loss with ablation knobs: use_dsr=False -> cls only; beta=1.0 -> tag-only DSR."""
    L_cls = F.cross_entropy(logits, y)
    if not use_dsr:
        return L_cls, {"loss": L_cls.item(), "cls": L_cls.item(),
                       "tag": 0.0, "rat": 0.0}
    L_tag = cosine_distance(z_tag_hat, z_tag)
    if beta >= 1.0 - 1e-6:
        L_dsr, rat_val = L_tag, 0.0
    else:
        L_rat = cosine_distance(d_hat, d)
        L_dsr = beta * L_tag + (1 - beta) * L_rat
        rat_val = L_rat.item()
    L = (1 - lam) * L_cls + lam * L_dsr
    return L, {"loss": L.item(), "cls": L_cls.item(),
               "tag": L_tag.item(), "rat": rat_val}


# Variant -> (fusion_mode, use_dsr, beta)
VARIANTS: Dict[str, Tuple[str, bool, float]] = {
    "full_GRACE":    ("dpcaf",        True,  0.6),
    "wo_gate":       ("dpcaf_nogate", True,  0.6),
    "wo_DPCAF":      ("elem_mul",     True,  0.6),
    "wo_rdm_target": ("dpcaf",        True,  1.0),   # tag-only DSR
    "wo_DSR":        ("dpcaf",        False, 0.6),   # cls only
}


def fit_grace_variant(dataset: str, seed: int, variant_name: str) -> Dict[str, float]:
    fusion_mode, use_dsr, beta = VARIANTS[variant_name]
    set_seed(seed)
    base = cfg.CACHE_ROOT / dataset

    def _ds(split):
        d = base / split
        return MemeDataset(
            ids   = json.loads((d / "ids.json").read_text()),
            x_img = torch.load(d / "x_img.pt"),
            x_txt = torch.load(d / "x_txt.pt"),
            z_tag = torch.load(d / "z_tag.pt"),
            d     = torch.load(d / "d.pt"),
            y     = torch.load(d / "y.pt"),
        )

    dl_tr = DataLoader(_ds("train"), batch_size=min(cfg.BATCH_SIZE, len(_ds("train"))), shuffle=True,
                       drop_last=False, num_workers=cfg.NUM_WORKERS)
    dl_va = DataLoader(_ds("val"),   batch_size=cfg.BATCH_SIZE, shuffle=False)
    dl_te = DataLoader(_ds("test"),  batch_size=cfg.BATCH_SIZE, shuffle=False)

    model = GRACEAblation(cfg.D_S, cfg.NUM_CLASSES, cfg.DROPOUT,
                          cfg.ARC_M, cfg.ARC_S, fusion_mode).to(cfg.DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS_GRACE)
    scaler = GradScaler(enabled=(cfg.DEVICE == "cuda"))

    best_auroc, best_state = -1.0, None
    for epoch in range(cfg.EPOCHS_GRACE):
        model.train()
        for batch in dl_tr:
            x_img = batch["x_img"].to(cfg.DEVICE)
            x_txt = batch["x_txt"].to(cfg.DEVICE)
            z_tag = batch["z_tag"].to(cfg.DEVICE)
            d_emb = batch["d"].to(cfg.DEVICE)
            y     = batch["y"].to(cfg.DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=(cfg.DEVICE == "cuda")):
                logits, z_hat, d_hat = model(x_img, x_txt, y)
                loss, _ = grace_loss_ablation(logits, z_hat, d_hat, y,
                                              z_tag, d_emb, cfg.LAMBDA, beta, use_dsr)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()
        val = evaluate(model, dl_va)
        if not math.isnan(val["auroc"]) and val["auroc"] > best_auroc:
            best_auroc = val["auroc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return evaluate(model, dl_te)


def run_ablation_study(dataset: str,
                       variants: Optional[List[str]] = None,
                       seeds: Optional[Sequence[int]] = None) -> pd.DataFrame:
    """Reproduce paper Table IV. Single seed default for time-economy; pass seeds=cfg.SEEDS for full."""
    variants = variants or list(VARIANTS.keys())
    seeds = list(seeds) if seeds is not None else [42]

    rows = []
    print(f"\n[{dataset}] Ablation study ({len(variants)} variants x {len(seeds)} seeds)")
    for v in variants:
        per_seed = [fit_grace_variant(dataset, s, v) for s in seeds]
        keys = per_seed[0].keys()
        m = {k: float(np.mean([r[k] for r in per_seed])) for k in keys}
        s = {k: float(np.std ([r[k] for r in per_seed])) for k in keys}
        rows.append({
            "variant":  v,
            "acc_%":    f"{m['acc']*100:.2f}" + (f" ± {s['acc']*100:.2f}" if len(seeds) > 1 else ""),
            "auroc_%":  f"{m['auroc']*100:.2f}" + (f" ± {s['auroc']*100:.2f}" if len(seeds) > 1 else ""),
            "f1_%":     f"{m['macro_f1']*100:.2f}" + (f" ± {s['macro_f1']*100:.2f}" if len(seeds) > 1 else ""),
        })
        print(f"  {v:18s}  auroc={m['auroc']:.4f}  acc={m['acc']:.4f}  f1={m['macro_f1']:.4f}")
    df = pd.DataFrame(rows)
    out_csv = cfg.OUTPUT_ROOT / f"ablation_{dataset}.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved -> {out_csv}")
    return df


# Example usage (uncomment after Stage 1-3 caches exist):
# ablation_hmc = run_ablation_study("hmc")
# ablation_hmc


## 9.6 Diagnostics & visualisations

For the BTech presentation: confusion matrix, ROC curve, and a grid of mispredicted memes (each with its image, OCR text, true label and `p(hate)`). PNGs are written to `/kaggle/working/` so you can right-click and download them straight from the Kaggle file browser.


In [71]:
from sklearn.metrics import confusion_matrix, roc_curve


@torch.no_grad()
def predict_with_model(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for batch in loader:
        logits, _, _ = model(batch["x_img"].to(cfg.DEVICE),
                             batch["x_txt"].to(cfg.DEVICE))
        all_logits.append(logits.cpu())
        all_y.append(batch["y"])
    logits = torch.cat(all_logits, dim=0)
    y = torch.cat(all_y, dim=0).numpy()
    probs = F.softmax(logits, dim=-1)[:, 1].numpy()
    preds = logits.argmax(dim=-1).numpy()
    return preds, probs, y


def plot_confusion(y, preds, title="Confusion Matrix"):
    cm = confusion_matrix(y, preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["non-hate", "hate"])
    ax.set_yticklabels(["non-hate", "hate"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    return fig


def plot_roc(y, probs, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y, probs)
    auc = roc_auc_score(y, probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    plt.tight_layout()
    return fig


def show_sample_predictions(dataset: str, model, n: int = 6, only: str = "errors"):
    """Show meme thumbnails with predicted/true labels. `only` in {'errors','correct','all'}."""
    base = cfg.CACHE_ROOT / dataset / "test"
    ds = MemeDataset(
        ids   = json.loads((base / "ids.json").read_text()),
        x_img = torch.load(base / "x_img.pt"),
        x_txt = torch.load(base / "x_txt.pt"),
        z_tag = torch.load(base / "z_tag.pt"),
        d     = torch.load(base / "d.pt"),
        y     = torch.load(base / "y.pt"),
    )
    dl = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
    preds, probs, y = predict_with_model(model, dl)

    paths = json.loads((base / "image_paths.json").read_text())
    texts = json.loads((base / "text.json").read_text())

    if only == "errors":
        idx = np.where(preds != y)[0]
    elif only == "correct":
        idx = np.where(preds == y)[0]
    else:
        idx = np.arange(len(y))
    if len(idx) == 0:
        print(f"No '{only}' samples")
        return None

    np.random.seed(0)
    pick = np.random.choice(idx, size=min(n, len(idx)), replace=False)
    cols = min(3, len(pick))
    rows = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4.5))
    axes = np.array(axes).reshape(-1)
    for ax, i in zip(axes, pick):
        try:
            ax.imshow(Image.open(paths[i]).convert("RGB"))
        except Exception as e:
            ax.text(0.5, 0.5, f"img error: {e}", ha="center")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"y={y[i]}  pred={preds[i]}  p(hate)={probs[i]:.2f}", fontsize=10)
        ax.set_xlabel(texts[i][:90], fontsize=8, wrap=True)
    for ax in axes[len(pick):]:
        ax.axis("off")
    plt.tight_layout()
    return fig


def run_diagnostics(dataset: str, seed: int = 42):
    """Train one GRACE seed and produce CM / ROC / sample-error PNGs."""
    set_seed(seed)
    base = cfg.CACHE_ROOT / dataset

    def _ds(split):
        d = base / split
        return MemeDataset(
            ids   = json.loads((d / "ids.json").read_text()),
            x_img = torch.load(d / "x_img.pt"),
            x_txt = torch.load(d / "x_txt.pt"),
            z_tag = torch.load(d / "z_tag.pt"),
            d     = torch.load(d / "d.pt"),
            y     = torch.load(d / "y.pt"),
        )
    dl_tr = DataLoader(_ds("train"), batch_size=min(cfg.BATCH_SIZE, len(_ds("train"))), shuffle=True,
                       drop_last=False, num_workers=cfg.NUM_WORKERS)
    dl_va = DataLoader(_ds("val"),   batch_size=cfg.BATCH_SIZE, shuffle=False)
    dl_te = DataLoader(_ds("test"),  batch_size=cfg.BATCH_SIZE, shuffle=False)

    model = GRACE(cfg.D_S, cfg.NUM_CLASSES, cfg.DROPOUT,
                  cfg.ARC_M, cfg.ARC_S).to(cfg.DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS_GRACE)
    scaler = GradScaler(enabled=(cfg.DEVICE == "cuda"))

    best_auroc, best_state = -1.0, None
    for epoch in range(cfg.EPOCHS_GRACE):
        model.train()
        for batch in dl_tr:
            x_img = batch["x_img"].to(cfg.DEVICE)
            x_txt = batch["x_txt"].to(cfg.DEVICE)
            z_tag = batch["z_tag"].to(cfg.DEVICE)
            d_emb = batch["d"].to(cfg.DEVICE)
            y     = batch["y"].to(cfg.DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=(cfg.DEVICE == "cuda")):
                logits, z_hat, d_hat = model(x_img, x_txt, y)
                loss, _ = grace_loss(logits, z_hat, d_hat, y,
                                     z_tag, d_emb, cfg.LAMBDA, cfg.BETA)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        sched.step()
        val = evaluate(model, dl_va)
        if not math.isnan(val["auroc"]) and val["auroc"] > best_auroc:
            best_auroc = val["auroc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)

    preds, probs, y = predict_with_model(model, dl_te)
    cm_fig = plot_confusion(y, preds, title=f"GRACE | {dataset} | confusion matrix")
    roc_fig = plot_roc(y, probs,       title=f"GRACE | {dataset} | ROC curve")
    err_fig = show_sample_predictions(dataset, model, n=6, only="errors")

    cm_fig.savefig(cfg.OUTPUT_ROOT / f"cm_{dataset}.png", dpi=140, bbox_inches="tight")
    roc_fig.savefig(cfg.OUTPUT_ROOT / f"roc_{dataset}.png", dpi=140, bbox_inches="tight")
    if err_fig is not None:
        err_fig.savefig(cfg.OUTPUT_ROOT / f"errors_{dataset}.png", dpi=140, bbox_inches="tight")
    print(f"PNGs saved under {cfg.OUTPUT_ROOT}")
    return model


# Example (uncomment after caches exist):
# diag_model = run_diagnostics("hmc")


## 10. Run the pipeline

The orchestrator `run_pipeline(ds)` runs every stage end-to-end for one dataset. Each stage is idempotent — if its cache file already exists it is skipped, so you can safely re-run after interruption.

**Recommended Kaggle workflow (free tier, 12-hour cap):**
1. **Always run the smoke-test cell first** (~10 min) to catch dataset-path errors cheaply.
2. Run `run_pipeline("hmc")` in Session 1 — LLaVA keyword/rationale generation takes 3–5 h.
3. Sessions 2–3: repeat for `"harmeme"` and `"pridemm"`.
4. Between sessions: go to *File → Add to Dataset* on `/kaggle/working/cache` so you can re-mount cached embeddings and skip LLaVA re-generation.


In [72]:
def run_pipeline(dataset: str) -> Dict:
    print(f"\n{'=' * 60}\n  Pipeline: {dataset}\n{'=' * 60}")
    # Stage 1a — CLIP features
    for split in ("train", "val", "test"):
        precompute_clip(dataset, split)
    # Stage 1b/c — keywords + encode
    for split in ("train", "val", "test"):
        precompute_keywords(dataset, split)
        encode_keywords(dataset, split)
    # Stage 2 — ATIN
    train_atin(dataset)
    precompute_z_tag(dataset)
    # Stage 1d/e — rationales + encode
    for split in ("train", "val", "test"):
        precompute_rationales(dataset, split)
        encode_rationales(dataset, split)
    # Stage 3 — RDM
    train_rdm(dataset)
    precompute_d(dataset)
    # Stage 4 — GRACE, 3 seeds
    return run_multi_seed(dataset)


### Step A — Smoke test (run this first, ~10 min)

Runs 50 samples end-to-end on HMC to verify dataset paths and model shapes before committing to a full LLaVA pass.


In [73]:
# ── SMOKE TEST ── run this first to catch path/format issues ──────────────────
# Save and switch cache roots so smoke-test data NEVER lands in the production cache.
_prod_cache = cfg.OUTPUT_ROOT / "cache"
_prod_ckpt  = cfg.OUTPUT_ROOT / "ckpt"
cfg.DEBUG_LIMIT = 50
cfg.CACHE_ROOT = cfg.OUTPUT_ROOT / "cache_debug"
cfg.CKPT_ROOT  = cfg.OUTPUT_ROOT / "ckpt_debug"
cfg.CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cfg.CKPT_ROOT.mkdir(parents=True, exist_ok=True)

smoke = run_pipeline("hmc")   # end-to-end dry run in ~10 min
print("Smoke test passed:", smoke)

# Restore production cache roots for the full run below.
cfg.DEBUG_LIMIT = None
cfg.CACHE_ROOT  = _prod_cache
cfg.CKPT_ROOT   = _prod_ckpt
cfg.CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cfg.CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Cache root reset -> {cfg.CACHE_ROOT}")



  Pipeline: hmc
[hmc/train] 50 samples


100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 89.8MiB/s]


CLIP hmc/train:   0%|          | 0/50 [00:00<?, ?it/s]

[hmc/train] saved x_img (50, 512)  x_txt (50, 512)
[hmc/val] 50 samples


CLIP hmc/val:   0%|          | 0/50 [00:00<?, ?it/s]

...

  seed=42  epoch  2/25  train_loss=21.6404  val_auroc=nan  val_acc=0.4000
  seed=42  epoch  3/25  train_loss=21.4390  val_auroc=nan  val_acc=0.4000
  seed=42  epoch  4/25  train_loss=21.3096  val_auroc=nan  val_acc=0.4000
  seed=42  epoch  5/25  train_loss=21.3894  val_auroc=nan  val_acc=0.0000
  seed=42  epoch  6/25  train_loss=10.7120  val_auroc=nan  val_acc=0.0000
  seed=42  epoch  7/25  train_loss=4.5666  val_auroc=nan  val_acc=0.0000
  seed=42  epoch  8/25  train_loss=4.5221  val_auroc=nan  val_acc=0.0000
  seed=42  epoch  9/25  train_loss=4.2937  val_auroc=nan  val_acc=0.0000
  seed=42  epoch 10/25  train_loss=3.8026  val_auroc=nan  val_acc=0.0000
  seed=42  epoch 11/25  train_loss=3.3636  val_auroc=nan  val_acc=0.0000
  seed=42  epoch 12/25  train_loss=2.7898  val_auroc=nan  val_acc=0.0000
  seed=42  epoch 13/25  train_loss=2.5280  val_auroc=nan  val_acc=0.0200
  seed=42  epoch 14/25  train_loss=2.2992  val_auroc=nan  val_acc=0.0200
  seed=42  epoch 15/25  train_loss=1.8939  val

/tmp/ipykernel_150/3590301875.py:80: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()  # per-epoch — T_max=EPOCHS_GRACE means one full cosine cycle over all epochs


  seed=123  epoch  2/25  train_loss=21.8191  val_auroc=nan  val_acc=1.0000
  seed=123  epoch  3/25  train_loss=21.9585  val_auroc=nan  val_acc=1.0000
  seed=123  epoch  4/25  train_loss=21.9148  val_auroc=nan  val_acc=1.0000
  seed=123  epoch  5/25  train_loss=22.0086  val_auroc=nan  val_acc=0.0000
  seed=123  epoch  6/25  train_loss=15.6115  val_auroc=nan  val_acc=0.0000
  seed=123  epoch  7/25  train_loss=5.9797  val_auroc=nan  val_acc=0.0000
  seed=123  epoch  8/25  train_loss=4.7572  val_auroc=nan  val_acc=0.0000
  seed=123  epoch  9/25  train_loss=4.5730  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 10/25  train_loss=4.7037  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 11/25  train_loss=4.3333  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 12/25  train_loss=4.0484  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 13/25  train_loss=3.7544  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 14/25  train_loss=3.2436  val_auroc=nan  val_acc=0.0000
  seed=123  epoch 15/25  train_lo

/tmp/ipykernel_150/3590301875.py:80: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()  # per-epoch — T_max=EPOCHS_GRACE means one full cosine cycle over all epochs


  seed=456  epoch  2/25  train_loss=19.9180  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  3/25  train_loss=19.7055  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  4/25  train_loss=20.0382  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  5/25  train_loss=19.8618  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  6/25  train_loss=17.6256  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  7/25  train_loss=9.6778  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  8/25  train_loss=5.8735  val_auroc=nan  val_acc=0.0000
  seed=456  epoch  9/25  train_loss=4.4829  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 10/25  train_loss=4.4637  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 11/25  train_loss=4.3042  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 12/25  train_loss=4.1655  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 13/25  train_loss=3.9119  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 14/25  train_loss=3.5995  val_auroc=nan  val_acc=0.0000
  seed=456  epoch 15/25  train_lo

### Step B — Full run (one dataset per session)

Uncomment **one** line per Kaggle session. LLaVA generation takes 3–5 h per dataset.


In [ ]:
# ── FULL RUN ── uncomment ONE line per Kaggle session ─────────────────────────
RESULTS: Dict[str, Dict] = {}

RESULTS["hmc"]     = run_pipeline("hmc")
# RESULTS["harmeme"] = run_pipeline("harmeme")
# RESULTS["pridemm"] = run_pipeline("pridemm")



  Pipeline: hmc
[hmc/train] 8500 samples


CLIP hmc/train:   0%|          | 0/8500 [00:00<?, ?it/s]

[hmc/train] saved x_img (8500, 512)  x_txt (8500, 512)
[hmc/val] 500 samples


CLIP hmc/val:   0%|          | 0/500 [00:00<?, ?it/s]

[hmc/val] saved x_img (500, 512)  x_txt (500, 512)
[hmc/test] 1000 samples


CLIP hmc/test:   0%|          | 0/1000 [00:00<?, ?it/s]

[hmc/test] saved x_img (1000, 512)  x_txt (1000, 512)
[hmc/train] generating keywords for 8500 new images


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Keywords hmc/train:   0%|          | 0/8500 [00:00<?, ?it/s]

## 11. Results summary & Paper Figures

Run these cells after `RESULTS` is populated. They generate all 5 paper figures from **actual model outputs** — not hardcoded values.


In [ ]:
def report():
    if not RESULTS:
        print("No results yet — run run_pipeline(...) for at least one dataset.")
        return
    rows = []
    for ds, r in RESULTS.items():
        m, s = r["mean"], r["std"]
        rows.append({
            "dataset":  ds,
            "accuracy": f"{m['acc']*100:.2f} ± {s['acc']*100:.2f}",
            "auroc":    f"{m['auroc']*100:.2f} ± {s['auroc']*100:.2f}",
            "macro_f1": f"{m['macro_f1']*100:.2f} ± {s['macro_f1']*100:.2f}",
        })
    print(pd.DataFrame(rows).to_string(index=False))

report()


### Fig 2 — HMC AUROC comparison (model result vs baselines)

In [ ]:
def plot_fig2_auroc(results: Dict, dataset: str = "hmc"):
    """Fig 2: bar chart comparing GRACE (actual result) vs paper baselines."""
    # Paper baseline numbers (Table IV)
    baselines = {
        "VisualBERT\nCOCO": 61.34,
        "MOMENTA":           62.66,
        "HateCLIPper":       76.46,
        "ISSUES":            79.03,
        "He et al.":         80.00,
        "MemeCLIP":          81.68,
        "MemeTAG":           82.17,
    }
    # Actual GRACE result from our run
    grace_auroc = results[dataset]["mean"]["auroc"] * 100

    methods = list(baselines.keys()) + ["GRACE\n(Ours)"]
    aurocs  = list(baselines.values()) + [grace_auroc]
    colors  = ["#90caf9"] * len(baselines) + ["#e53935"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(methods, aurocs, color=colors, edgecolor="white",
                  linewidth=0.8, width=0.6)
    for bar, val in zip(bars, aurocs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{val:.2f}", ha="center", va="bottom",
                fontsize=8.5, fontweight="bold")

    ax.axhline(82.65, color="gray", linestyle=":", lw=1.3)
    ax.text(len(methods) - 1.5, 82.85, "Human 82.65%", fontsize=7.5, color="gray")
    ax.set_ylim(55, max(aurocs) + 4)
    ax.set_ylabel("AUROC (%)", fontsize=11)
    ax.set_title(f"Fig 2: HMC Test-Seen AUROC — GRACE={grace_auroc:.2f}%",
                 fontsize=12, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5); ax.set_axisbelow(True)
    plt.tight_layout()
    out = cfg.OUTPUT_ROOT / "fig2_hmc_auroc.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out}")

plot_fig2_auroc(RESULTS)


### Fig 3 — Progressive ablation AUROC (from actual ablation run)

In [ ]:
def plot_fig3_ablation(dataset: str = "hmc"):
    """Fig 3: run ablation variants and plot progressive AUROC from real model."""
    print("Running ablation variants (1 seed each, ~20 min)...")
    abl_variants = ["wo_DSR", "wo_rdm_target", "wo_DPCAF", "full_GRACE"]
    abl_results  = run_ablation_study(dataset, variants=abl_variants, seeds=[42])

    # Map variant names to display labels
    label_map = {
        "wo_DSR":        "Base\n(no DSR)",
        "wo_rdm_target": "+RDM\ntarget",
        "wo_DPCAF":      "+DSR\n(elem-wise)",
        "full_GRACE":    "+DPCAF\n(Full)",
    }
    labels = [label_map[v] for v in abl_variants]
    aurocs = []
    for v in abl_variants:
        row = abl_results[abl_results["variant"] == v]
        val = float(str(row["auroc_%"].values[0]).split(" ")[0])
        aurocs.append(val)

    colors = ["#90caf9", "#42a5f5", "#1e88e5", "#0d47a1"]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(labels, aurocs, color=colors, edgecolor="white", width=0.5)
    for bar, val in zip(bars, aurocs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.2f}", ha="center", va="bottom",
                fontsize=9.5, fontweight="bold")
    ax.set_ylabel("AUROC (%)", fontsize=11)
    ax.set_title(f"Fig 3: Progressive AUROC Gain on {dataset.upper()}\n"
                 "Each GRACE component added progressively",
                 fontsize=11, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5); ax.set_axisbelow(True)
    plt.tight_layout()
    out = cfg.OUTPUT_ROOT / f"fig3_ablation_{dataset}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out}")

plot_fig3_ablation("hmc")


### Fig 4 — β sensitivity (actual model runs at different β values)

In [ ]:
def plot_fig4_beta_sensitivity(dataset: str = "hmc", seed: int = 42):
    """Fig 4: train GRACE at 7 beta values, plot val AUROC. ~2 min per beta."""
    betas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    val_aurocs = []

    base = cfg.CACHE_ROOT / dataset
    def _ds(split):
        d = base / split
        return MemeDataset(
            ids=json.loads((d/"ids.json").read_text()),
            x_img=torch.load(d/"x_img.pt"), x_txt=torch.load(d/"x_txt.pt"),
            z_tag=torch.load(d/"z_tag.pt"), d=torch.load(d/"d.pt"),
            y=torch.load(d/"y.pt"),
        )

    dl_va = DataLoader(_ds("val"), batch_size=cfg.BATCH_SIZE, shuffle=False)

    for beta in betas:
        set_seed(seed)
        dl_tr = DataLoader(_ds("train"),
                           batch_size=min(cfg.BATCH_SIZE, len(_ds("train"))),
                           shuffle=True, drop_last=False)
        model = GRACE(cfg.D_S, cfg.NUM_CLASSES, cfg.DROPOUT,
                      cfg.ARC_M, cfg.ARC_S).to(cfg.DEVICE)
        opt   = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                                  weight_decay=cfg.WEIGHT_DECAY)
        sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS_GRACE)
        scaler = GradScaler("cuda", enabled=(cfg.DEVICE=="cuda"))
        best_auroc = -1.0
        for epoch in range(cfg.EPOCHS_GRACE):
            model.train()
            for batch in dl_tr:
                x_img = batch["x_img"].to(cfg.DEVICE)
                x_txt = batch["x_txt"].to(cfg.DEVICE)
                z_tag = batch["z_tag"].to(cfg.DEVICE)
                d_emb = batch["d"].to(cfg.DEVICE)
                y     = batch["y"].to(cfg.DEVICE)
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(cfg.DEVICE=="cuda")):
                    logits, z_hat, d_hat = model(x_img, x_txt, y)
                    loss, _ = grace_loss(logits, z_hat, d_hat, y,
                                        z_tag, d_emb, cfg.LAMBDA, beta)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            sched.step()
            val = evaluate(model, dl_va)
            if not math.isnan(val["auroc"]) and val["auroc"] > best_auroc:
                best_auroc = val["auroc"]
        val_aurocs.append(best_auroc * 100)
        print(f"  beta={beta:.1f}  best_val_auroc={best_auroc*100:.2f}%")

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(betas, val_aurocs, "o-", color="#1565c0", lw=2,
            markersize=8, markerfacecolor="white", markeredgewidth=2)
    best_idx = int(np.argmax(val_aurocs))
    ax.plot(betas[best_idx], val_aurocs[best_idx], "o",
            color="#e53935", markersize=12, zorder=5)
    ax.annotate(f"β={betas[best_idx]}\n{val_aurocs[best_idx]:.2f}%",
                xy=(betas[best_idx], val_aurocs[best_idx]),
                xytext=(betas[best_idx]+0.1, val_aurocs[best_idx]+0.1),
                fontsize=9, color="#e53935",
                arrowprops=dict(arrowstyle="->", color="#e53935"))
    ax.set_xlabel("β  (keyword reconstruction weight)", fontsize=11)
    ax.set_ylabel("Val. AUROC (%)", fontsize=11)
    ax.set_title(f"Fig 4: DSR Weight β Sensitivity — {dataset.upper()}",
                 fontsize=11, fontweight="bold")
    ax.set_xticks(betas)
    ax.spines[["top","right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5); ax.set_axisbelow(True)
    plt.tight_layout()
    out = cfg.OUTPUT_ROOT / f"fig4_beta_{dataset}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out}")

plot_fig4_beta_sensitivity("hmc")


### Fig 5 — DPCAF attention heatmaps (real model, real meme images)

In [ ]:
def plot_fig5_attention(dataset: str = "hmc", seed: int = 42):
    """Fig 5: extract actual DPCAF attention weights for a hateful and a benign meme."""
    base = cfg.CACHE_ROOT / dataset / "test"
    ids   = json.loads((base/"ids.json").read_text())
    paths = json.loads((base/"image_paths.json").read_text())
    texts = json.loads((base/"text.json").read_text())
    x_img = torch.load(base/"x_img.pt")
    x_txt = torch.load(base/"x_txt.pt")
    z_tag = torch.load(base/"z_tag.pt")
    d_emb = torch.load(base/"d.pt")
    y_all = torch.load(base/"y.pt")

    # Load best trained model (seed 42)
    model = GRACE(cfg.D_S, cfg.NUM_CLASSES, cfg.DROPOUT,
                  cfg.ARC_M, cfg.ARC_S).to(cfg.DEVICE)

    # Re-train quickly to get model state (or load if ckpt exists)
    set_seed(seed)
    ds_tr = MemeDataset(
        ids=json.loads(((cfg.CACHE_ROOT/dataset/"train")/"ids.json").read_text()),
        x_img=torch.load(cfg.CACHE_ROOT/dataset/"train"/"x_img.pt"),
        x_txt=torch.load(cfg.CACHE_ROOT/dataset/"train"/"x_txt.pt"),
        z_tag=torch.load(cfg.CACHE_ROOT/dataset/"train"/"z_tag.pt"),
        d=torch.load(cfg.CACHE_ROOT/dataset/"train"/"d.pt"),
        y=torch.load(cfg.CACHE_ROOT/dataset/"train"/"y.pt"),
    )
    dl_tr = DataLoader(ds_tr, batch_size=min(cfg.BATCH_SIZE, len(ds_tr)),
                       shuffle=True, drop_last=False)
    opt   = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS_GRACE)
    scaler = GradScaler("cuda", enabled=(cfg.DEVICE=="cuda"))
    for epoch in range(cfg.EPOCHS_GRACE):
        model.train()
        for batch in dl_tr:
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(cfg.DEVICE=="cuda")):
                logits, zh, dh = model(batch["x_img"].to(cfg.DEVICE),
                                       batch["x_txt"].to(cfg.DEVICE),
                                       batch["y"].to(cfg.DEVICE))
                loss, _ = grace_loss(logits, zh, dh, batch["y"].to(cfg.DEVICE),
                                     batch["z_tag"].to(cfg.DEVICE),
                                     batch["d"].to(cfg.DEVICE), cfg.LAMBDA, cfg.BETA)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sched.step()
    model.eval()

    # Hook to capture DPCAF attention matrices
    _attn = {}
    def _hook(module, inp, out):
        _attn["A_i2t"] = module._last_A_i2t
        _attn["A_t2i"] = module._last_A_t2i

    # Monkey-patch DPCAF to store attention
    original_cross = model.dpcaf._cross
    def _cross_with_store(q, kv, key):
        scale = model.dpcaf.scale
        A = torch.bmm(q.unsqueeze(-1), kv.unsqueeze(-2)) * scale
        A = F.softmax(A, dim=-1)
        _attn[key] = A.detach().cpu()
        return torch.bmm(A, kv.unsqueeze(-1)).squeeze(-1)

    original_forward = model.dpcaf.forward
    def _forward_with_hooks(z_img, z_txt):
        z_img_ad = model.dpcaf.drop(z_img)
        z_txt_ad = model.dpcaf.drop(z_txt)
        a_i2t = _cross_with_store(z_img, z_txt, "A_i2t")
        a_t2i = _cross_with_store(z_txt, z_img, "A_t2i")
        f_dual = model.dpcaf.ln(model.dpcaf.proj_fusion(
            torch.cat([a_i2t, a_t2i], dim=-1)))
        g = torch.sigmoid(model.dpcaf.gate(torch.cat([z_img, z_txt], dim=-1)))
        residual = 0.5*(F.normalize(z_img,-1)+F.normalize(z_txt,-1))
        return model.dpcaf.drop(g*f_dual + (1-g)*residual)
    model.dpcaf.forward = _forward_with_hooks

    # Find one hateful and one benign sample from test
    hateful_idx = (y_all == 1).nonzero(as_tuple=True)[0][0].item()
    benign_idx  = (y_all == 0).nonzero(as_tuple=True)[0][0].item()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    cmap_h, cmap_b = "Reds", "Blues"

    for ax, idx, title, cmap, key in [
        (axes[0], hateful_idx, "(a) Hateful Meme — concentrated attention", cmap_h, "A_i2t"),
        (axes[1], benign_idx,  "(b) Benign Meme — diffuse attention",       cmap_b, "A_t2i"),
    ]:
        with torch.no_grad():
            _ = model(x_img[idx:idx+1].to(cfg.DEVICE),
                      x_txt[idx:idx+1].to(cfg.DEVICE))
        A = _attn[key][0].numpy()          # [512, 512]
        # Downsample to 16x16 for visualisation
        A16 = A[:16, :16]
        A16 = (A16 - A16.min()) / (A16.max() - A16.min() + 1e-8)
        im = ax.imshow(A16, cmap=cmap, aspect="auto")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title, fontsize=9.5, fontweight="bold")
        ax.set_xlabel("Feature dim (Text side)", fontsize=8)
        ax.set_ylabel("Feature dim (Image side)", fontsize=8)
        caption = texts[idx][:60] + "..." if len(texts[idx]) > 60 else texts[idx]
        ax.text(0.02, 0.97, f'"{caption}"', transform=ax.transAxes,
                fontsize=7, va="top",
                bbox=dict(boxstyle="round", fc="white", alpha=0.8))

    fig.suptitle("Fig 5: DPCAF Attention Weights (Real Model)", fontsize=12,
                 fontweight="bold")
    plt.tight_layout()
    out = cfg.OUTPUT_ROOT / f"fig5_attention_{dataset}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {out}")

plot_fig5_attention("hmc")


---

### Reproducibility notes

- The paper specifies seeds `{42, 123, 456}` and reports the mean across them — `run_multi_seed` does the same.
- All hyperparameters in `CFG` match Section IV.C verbatim; do **not** edit them when reproducing.
- `cudnn.deterministic=True` and `cudnn.benchmark=False` are set inside `set_seed`. Some non-determinism from `torch.acos` in ArcFace remains unavoidable.
- The implementation of DPCAF uses **feature-level outer-product attention** — the literal interpretation of Eqs. 8–9 with single 512-d vectors per modality (analogous to HateCLIPper's interaction matrix). Each attention matrix is `[B, 512, 512]`; total memory per batch (64) ≈ 134 MB which comfortably fits on T4.
- Mixed precision is enabled in Stage 4 only; ArcFace upcasts to fp32 internally for arccos stability.

### Known portability issues to watch for on Kaggle

- `bitsandbytes` requires a CUDA build matching the Kaggle PyTorch image; if the install above errors, drop the version pin (`bitsandbytes` alone).
- `clip.tokenize` truncates to 77 tokens — long meme captions are clipped, matching the paper's setup.
- If HMC `parthplc/...` is unavailable, use Meta's official DrivenData release and upload as a private dataset.
